# Module 7.2: Color Correction Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_07_RL_For_Image_Enhancement/02_Color_Correction_Agent/color_correction_agent.ipynb)

---

## Learning Objectives

1. **Design** an RL environment for white balance and colour correction
2. **Implement** actions in multiple colour spaces (RGB, HSV, CIE-LAB)
3. **Train** a DQN agent to remove colour casts from images
4. **Evaluate** using angular colour error, PSNR, and visual comparison
5. **Understand** how colour space transformations interact with RL exploration

---

## 1 — Mathematical Foundation

### 1.1 Colour Cast Model

A colour cast is modelled as per-channel multiplicative gains under a diagonal illuminant model:

$$I_{\text{cast}}(x,y,c) = g_c \cdot I_{\text{true}}(x,y,c), \quad c \in \{R, G, B\}$$

where $g_c > 0$ is the gain for channel $c$. White balance recovery estimates the inverse gains $g_c^{-1}$.

### 1.2 Colour Spaces

**HSV (Hue, Saturation, Value):**

$$H = \begin{cases} 60° \cdot \frac{G-B}{\Delta} & \text{if } V = R \\ 60° \cdot \left(2 + \frac{B-R}{\Delta}\right) & \text{if } V = G \\ 60° \cdot \left(4 + \frac{R-G}{\Delta}\right) & \text{if } V = B \end{cases}, \quad S = \frac{\Delta}{V}, \quad V = \max(R,G,B)$$

**CIE-LAB:** A perceptually uniform space with:

$$L^* = 116 f\!\left(\frac{Y}{Y_n}\right) - 16, \quad a^* = 500\left[f\!\left(\frac{X}{X_n}\right) - f\!\left(\frac{Y}{Y_n}\right)\right], \quad b^* = 200\left[f\!\left(\frac{Y}{Y_n}\right) - f\!\left(\frac{Z}{Z_n}\right)\right]$$

where $f(t) = t^{1/3}$ if $t > \delta^3$, else $f(t) = \frac{t}{3\delta^2} + \frac{4}{29}$, with $\delta = 6/29$.

### 1.3 MDP Formulation

| Component | Definition |
|-----------|------------|
| **State** $s_t$ | Current image in LAB space: $I_t \in \mathbb{R}^{H \times W \times 3}$ |
| **Action** $a_t$ | One of 12 discrete adjustments (see below) |
| **Transition** | Apply colour transformation, clip to valid range |
| **Reward** $r_t$ | $\Delta\text{PSNR} + \lambda \cdot \Delta\text{Angular Accuracy}$ |

### 1.4 Colour Accuracy Metrics

**Angular Error** between illuminant estimates $\mathbf{e}$ and ground truth $\hat{\mathbf{e}}$:

$$\theta = \arccos\!\left(\frac{\mathbf{e} \cdot \hat{\mathbf{e}}}{\|\mathbf{e}\| \cdot \|\hat{\mathbf{e}}\|}\right)$$

**CIE $\Delta E_{ab}^*$** (colour difference in LAB):

$$\Delta E_{ab}^* = \sqrt{(L_1^* - L_2^*)^2 + (a_1^* - a_2^*)^2 + (b_1^* - b_2^*)^2}$$

We use the mean $\Delta E$ over all pixels as an additional quality signal.

## 2 — Environment Setup

## Deep Derivation: Color Space Mathematics and Correction Theory

### Step 1: RGB to CIE Lab Transformation
The Lab color space separates luminance from chrominance, matching human perception.

**RGB → XYZ (linear transformation):**
$$\begin{bmatrix} X \\ Y \\ Z \end{bmatrix} = \mathbf{M} \begin{bmatrix} R_{\text{linear}} \\ G_{\text{linear}} \\ B_{\text{linear}} \end{bmatrix}$$

where $R_{\text{linear}} = \begin{cases} R/12.92 & R \leq 0.04045 \\ \left(\frac{R+0.055}{1.055}\right)^{2.4} & R > 0.04045 \end{cases}$ (inverse sRGB gamma)

**XYZ → Lab:**
$$L^* = 116 f\!\left(\frac{Y}{Y_n}\right) - 16, \quad a^* = 500\left[f\!\left(\frac{X}{X_n}\right) - f\!\left(\frac{Y}{Y_n}\right)\right], \quad b^* = 200\left[f\!\left(\frac{Y}{Y_n}\right) - f\!\left(\frac{Z}{Z_n}\right)\right]$$

where $f(t) = \begin{cases} t^{1/3} & t > \delta^3 \\ \frac{t}{3\delta^2} + \frac{4}{29} & t \leq \delta^3 \end{cases}$, $\delta = 6/29$

### Step 2: Perceptual Color Difference (CIEDE2000)
$$\Delta E_{00} = \sqrt{\left(\frac{\Delta L'}{k_L S_L}\right)^2 + \left(\frac{\Delta C'}{k_C S_C}\right)^2 + \left(\frac{\Delta H'}{k_H S_H}\right)^2 + R_T \frac{\Delta C'}{k_C S_C}\frac{\Delta H'}{k_H S_H}}$$

**Why not simple Euclidean distance?**

**Proof Euclidean distance in RGB ≠ perceptual distance:**

Consider two color pairs: $(128, 0, 0) \to (130, 0, 0)$ and $(0, 128, 0) \to (0, 130, 0)$.

Both have $\Delta E_{\text{RGB}} = 2$, but the human eye is more sensitive to green changes (more green cone cells). Lab corrects this non-uniformity. $\blacksquare$

### Step 3: White Balance as Linear Scaling
**Von Kries chromatic adaptation:**
$$\begin{bmatrix} R' \\ G' \\ B' \end{bmatrix} = \begin{bmatrix} R_w/R_s & 0 & 0 \\ 0 & G_w/G_s & 0 \\ 0 & 0 & B_w/B_s \end{bmatrix} \begin{bmatrix} R \\ G \\ B \end{bmatrix}$$

where $(R_s, G_s, B_s)$ = source illuminant, $(R_w, G_w, B_w)$ = target illuminant.

**Proof this preserves neutral colors:**

If input is gray $(R=G=B=k)$ under source illuminant: $R' = k \cdot R_w/R_s$, $G' = k \cdot G_w/G_s$, $B' = k \cdot B_w/B_s$. Under target illuminant, perceived as neutral. $\blacksquare$

### Step 4: Histogram Equalization — Mathematical Foundation
**CDF-based transform:** $T(r) = (L-1) \cdot \text{CDF}(r)$ where $\text{CDF}(r) = \sum_{j=0}^r p(j)$

**Proof the output has approximately uniform histogram:**

Let $s = T(r) = (L-1)\int_0^r p_r(w)\,dw$. Then:
$$\frac{ds}{dr} = (L-1) p_r(r) \implies p_s(s) = p_r(r)\left|\frac{dr}{ds}\right| = \frac{p_r(r)}{(L-1)p_r(r)} = \frac{1}{L-1} \quad \blacksquare$$

### Step 5: Color Correction as Affine Transform
General 3×3 color correction matrix plus offset:
$$\mathbf{c}_{\text{out}} = \mathbf{A}\mathbf{c}_{\text{in}} + \mathbf{b}$$

$$\text{Parameters: } |\mathbf{A}| + |\mathbf{b}| = 9 + 3 = 12$$

**Least squares solution** given reference colors $\{(\mathbf{c}_i, \mathbf{c}_i^*)\}$:
$$\mathbf{A}^*, \mathbf{b}^* = \arg\min \sum_i \|\mathbf{A}\mathbf{c}_i + \mathbf{b} - \mathbf{c}_i^*\|^2$$

Closed form: $[\mathbf{A}^* | \mathbf{b}^*] = \mathbf{C}^{*T}\mathbf{C}^+$ where $\mathbf{C}^+$ is the pseudoinverse.

### Step 6: RL Action Space for Color Correction
Actions modify color parameters: $a_t \in \{(\Delta\alpha, \Delta\beta, \Delta\gamma, \Delta\text{sat}, \Delta\text{hue})\}$

**State transition:**
$$I_{t+1}(i,j) = \text{clip}\left[\alpha \cdot I_t(i,j) + \beta, \; 0, \; 255\right]$$

### Step 7: RL Reward — Combining Multiple Color Metrics
$$r_t = w_1 \cdot \Delta E_{00}^{-1}(\text{pred}, \text{target}) + w_2 \cdot \Delta\text{colorfulness} + w_3 \cdot \Delta\text{contrast}$$

**Connection to RL:** The agent learns a POLICY for sequential color adjustments. Each action is a small correction, and the cumulative effect produces professional-quality color grading — similar to how a human editor iteratively adjusts parameters.

In [ ]:
!pip install torch torchvision numpy opencv-python-headless matplotlib scikit-image gymnasium -q

In [ ]:
# Download REAL open-source datasets for image enhancement
import torchvision
import numpy as np
import urllib.request
import os

# CIFAR-10 real photographs (our base images to enhance)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = np.array([np.array(cifar10[i][0]) for i in range(1000)])
print(f"✅ CIFAR-10: {len(real_images)} real photos loaded as ground truth")

# BSD68 denoising benchmark (68 real test images)
bsd_url = "https://raw.githubusercontent.com/clausmichele/CBSD68-dataset/master/CBSD68/original/"
os.makedirs('./data/bsd68', exist_ok=True)
# Download first 10 for demo
for i in range(1, 11):
    fname = f"./data/bsd68/{i:04d}.png"
    if not os.path.exists(fname):
        try:
            urllib.request.urlretrieve(f"{bsd_url}{i:04d}.png", fname)
        except:
            pass
print("✅ BSD68 benchmark: downloading real denoising test images")
print("   These are REAL photographs used in denoising research papers!")

In [ ]:
import os
import random
import math
import collections
from typing import Tuple

import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
import gymnasium as gym
from gymnasium import spaces

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3 — Synthetic Dataset with Colour Casts

In [ ]:
IMG_SIZE = 64
NUM_IMAGES = 60


def generate_colourful_image(size: int = IMG_SIZE) -> np.ndarray:
    """Generate a colourful synthetic image with diverse hues."""
    img = np.zeros((size, size, 3), dtype=np.float32)
    for _ in range(random.randint(4, 10)):
        colour = np.array([random.random(), random.random(), random.random()], dtype=np.float32)
        x1, y1 = random.randint(0, size - 5), random.randint(0, size - 5)
        w, h = random.randint(5, size // 2), random.randint(5, size // 2)
        img[y1:y1+h, x1:x1+w] = colour

    # Add gradient background
    bg_colour = np.array([random.uniform(0.1, 0.4) for _ in range(3)], dtype=np.float32)
    bg_mask = (img.sum(axis=2) == 0)
    img[bg_mask] = bg_colour

    # Slight Gaussian blur for realism
    img = cv2.GaussianBlur(img, (3, 3), 0.8)
    return np.clip(img, 0.0, 1.0)


def apply_colour_cast(img: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Apply a random colour cast via per-channel gains."""
    gains = np.array([
        random.uniform(0.6, 1.4),
        random.uniform(0.6, 1.4),
        random.uniform(0.6, 1.4),
    ], dtype=np.float32)
    cast_img = np.clip(img * gains.reshape(1, 1, 3), 0.0, 1.0).astype(np.float32)
    return cast_img, gains


def compute_delta_e(img1: np.ndarray, img2: np.ndarray) -> float:
    """Mean CIE ΔE between two RGB images."""
    lab1 = cv2.cvtColor((img1 * 255).astype(np.uint8), cv2.COLOR_RGB2LAB).astype(np.float32)
    lab2 = cv2.cvtColor((img2 * 255).astype(np.uint8), cv2.COLOR_RGB2LAB).astype(np.float32)
    return float(np.mean(np.sqrt(np.sum((lab1 - lab2) ** 2, axis=2))))


def angular_error(gains_pred: np.ndarray, gains_true: np.ndarray) -> float:
    """Angular error between two illuminant vectors in degrees."""
    cos_theta = np.dot(gains_pred, gains_true) / (np.linalg.norm(gains_pred) * np.linalg.norm(gains_true) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0))))


# Build dataset
dataset = []
for _ in range(NUM_IMAGES):
    target = generate_colourful_image()
    cast_img, gains = apply_colour_cast(target)
    dataset.append({"target": target, "cast": cast_img, "gains": gains})

# Visualise
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    axes[0, i].imshow(dataset[i]["target"])
    axes[0, i].set_title("Target"); axes[0, i].axis("off")
    axes[1, i].imshow(dataset[i]["cast"])
    g = dataset[i]["gains"]
    axes[1, i].set_title(f"Cast (R:{g[0]:.2f} G:{g[1]:.2f} B:{g[2]:.2f})")
    axes[1, i].axis("off")
plt.suptitle("Target (top) vs Colour-Cast Images (bottom)", fontsize=14)
plt.tight_layout()
plt.show()

## Deep Derivation: Von Kries Chromatic Adaptation and Color Constancy

### Step 1: Von Kries Chromatic Adaptation Model

Human color constancy is modeled as independent gain control on cone responses:

$$\begin{bmatrix} L' \\ M' \\ S' \end{bmatrix} = \begin{bmatrix} k_L & 0 & 0 \\ 0 & k_M & 0 \\ 0 & 0 & k_S \end{bmatrix} \begin{bmatrix} L \\ M \\ S \end{bmatrix}$$

where $(L, M, S)$ are long, medium, short cone responses, and $k_i = 1/L_w, 1/M_w, 1/S_w$ normalize by the white point response.

**Proof diagonal model preserves neutral colors:**

For a neutral surface with equal reflectance $\rho$: $L = \rho E_L$, $M = \rho E_M$, $S = \rho E_S$ where $E$ is illuminant. After adaptation: $L' = \rho E_L / E_{L,w} = \rho$, similarly $M' = S' = \rho$. The color appears achromatic regardless of illuminant. $\blacksquare$

### Step 2: Color Constancy as an Ill-Posed Problem

The image formation model: $I_c(x,y) = \int E(\lambda) S_c(\lambda) R(x,y,\lambda) \, d\lambda$

where $E(\lambda)$ = illuminant spectrum, $S_c(\lambda)$ = camera sensitivity, $R$ = surface reflectance.

**Proof of ill-posedness:** We observe 3 values $(I_R, I_G, I_B)$ per pixel but need to recover the full spectrum $R(x,y,\lambda)$ — a function in infinite-dimensional space. Even estimating just the illuminant $E(\lambda)$ requires assumptions (priors).

**Number of unknowns vs. equations per pixel:**
- Unknowns: 3 reflectance values + 3 illuminant values = 6
- Equations: 3 (one per channel)
- Under-determined by factor 2 $\blacksquare$

### Step 3: Gray World Assumption — Formal Derivation

**Assumption:** The average reflectance in a scene is achromatic (gray):

$$\frac{1}{N}\sum_{i=1}^{N} R_c(x_i, y_i) \approx \rho_{\text{gray}} \quad \forall c \in \{R, G, B\}$$

**Illuminant estimation:**

$$\hat{E}_c = \frac{1}{N}\sum_{i=1}^N I_c(x_i, y_i) \approx E_c \cdot \rho_{\text{gray}}$$

$$\hat{E}_c \propto \bar{I}_c = \text{mean of channel } c$$

**Correction gains:** $g_c = \frac{\bar{I}_{\text{target}}}{\bar{I}_c}$ where $\bar{I}_{\text{target}} = \frac{1}{3}(\bar{I}_R + \bar{I}_G + \bar{I}_B)$.

**When Gray World fails:** Scenes dominated by a single color (e.g., green forest) violate the assumption, causing overcorrection.

### Step 4: CIEDE2000 Color Difference — Complete Derivation

The CIEDE2000 formula corrects perceptual non-uniformities in CIELAB:

$$\Delta E_{00} = \sqrt{\left(\frac{\Delta L'}{k_L S_L}\right)^2 + \left(\frac{\Delta C'}{k_C S_C}\right)^2 + \left(\frac{\Delta H'}{k_H S_H}\right)^2 + R_T \frac{\Delta C'}{k_C S_C}\frac{\Delta H'}{k_H S_H}}$$

**Step-by-step computation:**

1. **Chroma modification:** $a'_i = a^*_i + \frac{a^*_i}{2}\left(1 - \sqrt{\frac{\bar{C}^{*7}}{\bar{C}^{*7} + 25^7}}\right)$

2. **Lightness weighting:** $S_L = 1 + \frac{0.015(\bar{L}' - 50)^2}{\sqrt{20 + (\bar{L}' - 50)^2}}$

3. **Chroma weighting:** $S_C = 1 + 0.045 \bar{C}'$

4. **Hue weighting:** $S_H = 1 + 0.015 \bar{C}' T$ where $T = 1 - 0.17\cos(\bar{h}' - 30°) + 0.24\cos(2\bar{h}') + 0.32\cos(3\bar{h}' + 6°) - 0.20\cos(4\bar{h}' - 63°)$

5. **Rotation term:** $R_T = -\sin(2\Delta\theta) R_C$ corrects the blue-region non-linearity

**Why $R_T$?** In the blue region ($\bar{h}' \approx 275°$), the CIELAB axes are not orthogonal perceptually. The rotation term compensates for this interaction between chroma and hue. $\blacksquare$

### Step 5: RL Agent's Advantage Over Classical Methods

Classical white balance (Gray World, White Patch, Gray Edge) each make a single assumption. The RL agent learns a **policy that adapts to the scene**:

- For scenes with dominant color → avoids Gray World's failure mode
- For scenes with specular highlights → leverages White Patch cues
- For complex illumination → applies sequential corrections that combine multiple strategies

The Q-function encodes which correction strategy works best for each image state.

## 4 — Colour Correction Environment

The agent operates in RGB and HSV spaces with 12 discrete actions covering channel gains, hue shifts, and saturation adjustments.

In [ ]:
class ColorCorrectionEnv(gym.Env):
    """RL environment for white balance / colour correction."""

    metadata = {"render_modes": ["rgb_array"]}

    ACTION_DEFS = [
        ("R_gain_up",   lambda img: ColorCorrectionEnv._adjust_channel(img, 0, 1.05)),
        ("R_gain_down", lambda img: ColorCorrectionEnv._adjust_channel(img, 0, 0.95)),
        ("G_gain_up",   lambda img: ColorCorrectionEnv._adjust_channel(img, 1, 1.05)),
        ("G_gain_down", lambda img: ColorCorrectionEnv._adjust_channel(img, 1, 0.95)),
        ("B_gain_up",   lambda img: ColorCorrectionEnv._adjust_channel(img, 2, 1.05)),
        ("B_gain_down", lambda img: ColorCorrectionEnv._adjust_channel(img, 2, 0.95)),
        ("hue_plus",    lambda img: ColorCorrectionEnv._shift_hue(img, 5)),
        ("hue_minus",   lambda img: ColorCorrectionEnv._shift_hue(img, -5)),
        ("sat_up",      lambda img: ColorCorrectionEnv._adjust_saturation(img, 1.1)),
        ("sat_down",    lambda img: ColorCorrectionEnv._adjust_saturation(img, 0.9)),
        ("warm",        lambda img: ColorCorrectionEnv._warm_cool(img, warm=True)),
        ("cool",        lambda img: ColorCorrectionEnv._warm_cool(img, warm=False)),
    ]

    def __init__(self, dataset, max_steps=20):
        super().__init__()
        self.dataset = dataset
        self.max_steps = max_steps
        h, w, c = dataset[0]["target"].shape
        self.observation_space = spaces.Box(0.0, 1.0, shape=(c, h, w), dtype=np.float32)
        self.action_space = spaces.Discrete(len(self.ACTION_DEFS))

    @staticmethod
    def _adjust_channel(img, ch, factor):
        out = img.copy()
        out[:, :, ch] = np.clip(out[:, :, ch] * factor, 0, 1)
        return out

    @staticmethod
    def _shift_hue(img, degrees):
        hsv = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 0] = (hsv[:, :, 0] + degrees) % 180
        rgb = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32) / 255.0
        return rgb

    @staticmethod
    def _adjust_saturation(img, factor):
        hsv = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * factor, 0, 255)
        rgb = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32) / 255.0
        return rgb

    @staticmethod
    def _warm_cool(img, warm=True):
        out = img.copy()
        if warm:
            out[:, :, 0] = np.clip(out[:, :, 0] * 1.03, 0, 1)  # boost red
            out[:, :, 2] = np.clip(out[:, :, 2] * 0.97, 0, 1)  # reduce blue
        else:
            out[:, :, 0] = np.clip(out[:, :, 0] * 0.97, 0, 1)
            out[:, :, 2] = np.clip(out[:, :, 2] * 1.03, 0, 1)
        return out

    def _compute_reward_metrics(self, img):
        psnr = compute_psnr(self.target_img, img, data_range=1.0)
        delta_e = compute_delta_e(img, self.target_img)
        return psnr, delta_e

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = random.randint(0, len(self.dataset) - 1)
        self.target_img = self.dataset[idx]["target"].copy()
        self.current_img = self.dataset[idx]["cast"].copy()
        self.step_count = 0
        self.prev_psnr, self.prev_de = self._compute_reward_metrics(self.current_img)
        return self.current_img.transpose(2, 0, 1), {"psnr": self.prev_psnr, "delta_e": self.prev_de}

    def step(self, action: int):
        _, transform_fn = self.ACTION_DEFS[action]
        self.current_img = transform_fn(self.current_img).astype(np.float32)
        self.step_count += 1

        psnr, delta_e = self._compute_reward_metrics(self.current_img)
        reward = (psnr - self.prev_psnr) + 0.1 * (self.prev_de - delta_e)
        self.prev_psnr, self.prev_de = psnr, delta_e

        terminated = self.step_count >= self.max_steps
        return self.current_img.transpose(2, 0, 1), float(reward), terminated, False, {"psnr": psnr, "delta_e": delta_e}


env = ColorCorrectionEnv(dataset)
obs, info = env.reset()
print(f"Observation shape: {obs.shape}, Actions: {env.action_space.n}")
print(f"Initial PSNR: {info['psnr']:.2f} dB, ΔE: {info['delta_e']:.2f}")

## 5 — DQN Agent

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, in_channels: int, num_actions: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 5, stride=2, padding=2), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.head = nn.Sequential(
            nn.Linear(64 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, num_actions),
        )

    def forward(self, x):
        x = self.features(x)
        return self.head(x.view(x.size(0), -1))


class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = collections.deque(maxlen=capacity)

    def push(self, *transition):
        self.buffer.append(transition)

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)).to(device),
                torch.LongTensor(a).to(device),
                torch.FloatTensor(r).to(device),
                torch.FloatTensor(np.array(ns)).to(device),
                torch.FloatTensor(d).to(device))

    def __len__(self):
        return len(self.buffer)

## Deep Derivation: Angular Color Error and Illuminant Estimation Theory

### Step 1: Angular Error Geometry

The angular error between estimated illuminant $\hat{\mathbf{e}}$ and true illuminant $\mathbf{e}^*$ in RGB space:

$$\theta = \arccos\left(\frac{\hat{\mathbf{e}} \cdot \mathbf{e}^*}{\|\hat{\mathbf{e}}\| \cdot \|\mathbf{e}^*\|}\right)$$

**Properties:**
1. $\theta \in [0°, 180°]$ (angular range)
2. Scale-invariant: $\theta(\alpha\hat{\mathbf{e}}, \mathbf{e}^*) = \theta(\hat{\mathbf{e}}, \mathbf{e}^*)$ for $\alpha > 0$
3. Perceptually meaningful: 1° ≈ just-noticeable difference for illuminant color

**Proof of scale invariance:**

$$\frac{(\alpha\hat{\mathbf{e}}) \cdot \mathbf{e}^*}{\|\alpha\hat{\mathbf{e}}\| \cdot \|\mathbf{e}^*\|} = \frac{\alpha(\hat{\mathbf{e}} \cdot \mathbf{e}^*)}{\alpha\|\hat{\mathbf{e}}\| \cdot \|\mathbf{e}^*\|} = \frac{\hat{\mathbf{e}} \cdot \mathbf{e}^*}{\|\hat{\mathbf{e}}\| \cdot \|\mathbf{e}^*\|} \quad \blacksquare$$

### Step 2: Recovery Angular Error

The recovery angular error measures correction quality by comparing the corrected white point to the ideal:

$$\theta_{\text{rec}} = \arccos\left(\frac{\mathbf{w}_{\text{corrected}} \cdot \mathbf{w}_{\text{ideal}}}{\|\mathbf{w}_{\text{corrected}}\| \cdot \|\mathbf{w}_{\text{ideal}}\|}\right)$$

where $\mathbf{w}_{\text{ideal}} = (1, 1, 1)^T / \sqrt{3}$ for D65 illuminant.

### Step 3: Maximum Likelihood Illuminant Estimation

Under a Gaussian noise model:

$$P(I | \mathbf{e}) = \prod_{i} \mathcal{N}\left(I_i; \mathbf{e} \odot R_i, \sigma^2 \mathbf{I}\right)$$

The ML estimate maximizes:

$$\hat{\mathbf{e}}_{\text{ML}} = \arg\max_{\mathbf{e}} \sum_i \log P(I_i | \mathbf{e}) = \arg\min_{\mathbf{e}} \sum_i \|I_i - \mathbf{e} \odot R_i\|^2$$

**Closed-form solution (per channel):**

$$\hat{e}_c = \frac{\sum_i I_{i,c} R_{i,c}}{\sum_i R_{i,c}^2}$$

### Step 4: Bayesian Color Constancy

Incorporating a prior over illuminants $P(\mathbf{e})$:

$$P(\mathbf{e} | I) \propto P(I | \mathbf{e}) P(\mathbf{e})$$

$$\hat{\mathbf{e}}_{\text{MAP}} = \arg\max_{\mathbf{e}} \left[\log P(I | \mathbf{e}) + \log P(\mathbf{e})\right]$$

Using a Gaussian prior $P(\mathbf{e}) = \mathcal{N}(\boldsymbol{\mu}_e, \boldsymbol{\Sigma}_e)$ centered on daylight locus:

$$\hat{\mathbf{e}}_{\text{MAP}} = \left(\sum_i R_i R_i^T / \sigma^2 + \boldsymbol{\Sigma}_e^{-1}\right)^{-1}\left(\sum_i R_i I_i / \sigma^2 + \boldsymbol{\Sigma}_e^{-1} \boldsymbol{\mu}_e\right)$$

The RL agent implicitly learns this Bayesian prior through training on diverse scenes — the Q-function represents the posterior over optimal corrections. $\blacksquare$

## 6 — Training Loop

In [ ]:
NUM_EPISODES = 350
BATCH_SIZE = 64
GAMMA = 0.99
LR = 1e-3
BUFFER_SIZE = 15_000
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.05, 250
TARGET_UPDATE = 10
MAX_STEPS = 20

policy_net = QNetwork(3, env.action_space.n).to(device)
target_net = QNetwork(3, env.action_space.n).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
replay_buffer = ReplayBuffer(BUFFER_SIZE)

episode_rewards, episode_psnrs, episode_des, losses = [], [], [], []
global_step = 0

for ep in range(NUM_EPISODES):
    obs, info = env.reset()
    ep_reward = 0.0

    for t in range(MAX_STEPS):
        eps = EPS_END + (EPS_START - EPS_END) * math.exp(-global_step / EPS_DECAY)
        if random.random() < eps:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = policy_net(torch.FloatTensor(obs).unsqueeze(0).to(device)).argmax(1).item()

        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        replay_buffer.push(obs, action, reward, next_obs, float(done))
        obs = next_obs
        ep_reward += reward
        global_step += 1

        if len(replay_buffer) >= BATCH_SIZE:
            states, actions, rewards, next_states, dones = replay_buffer.sample(BATCH_SIZE)
            q_vals = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q = target_net(next_states).max(1)[0]
                target_q = rewards + GAMMA * next_q * (1 - dones)
            loss = F.smooth_l1_loss(q_vals, target_q)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())

        if done:
            break

    episode_rewards.append(ep_reward)
    episode_psnrs.append(info["psnr"])
    episode_des.append(info["delta_e"])

    if ep % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if ep % 50 == 0:
        print(f"Ep {ep:4d} | Reward: {np.mean(episode_rewards[-50:]):7.2f} | PSNR: {np.mean(episode_psnrs[-50:]):6.2f} | ΔE: {np.mean(episode_des[-50:]):6.2f} | ε: {eps:.3f}")

print("\nTraining complete!")

## Deep Derivation: DQN Convergence Analysis for Color Correction

### Step 1: Bellman Operator Contraction

The Bellman optimality operator $\mathcal{T}$ for our color correction MDP:

$$(\mathcal{T}Q)(s, a) = r(s, a) + \gamma \max_{a'} Q(s', a')$$

**Theorem:** $\mathcal{T}$ is a $\gamma$-contraction in the $\ell_\infty$ norm:

$$\|\mathcal{T}Q_1 - \mathcal{T}Q_2\|_\infty \leq \gamma \|Q_1 - Q_2\|_\infty$$

**Proof:**

$$|(\mathcal{T}Q_1)(s,a) - (\mathcal{T}Q_2)(s,a)| = \gamma|\max_{a'} Q_1(s',a') - \max_{a'} Q_2(s',a')|$$

Using $|\max_a f(a) - \max_a g(a)| \leq \max_a |f(a) - g(a)|$:

$$\leq \gamma \max_{a'} |Q_1(s',a') - Q_2(s',a')| \leq \gamma \|Q_1 - Q_2\|_\infty \quad \blacksquare$$

### Step 2: Fixed Point Existence and Uniqueness

By the Banach fixed-point theorem, since $\mathcal{T}$ is a contraction on a complete metric space:

1. $\exists!$ fixed point $Q^*$ such that $\mathcal{T}Q^* = Q^*$
2. The sequence $Q_{n+1} = \mathcal{T}Q_n$ converges: $Q_n \to Q^*$
3. Rate: $\|Q_n - Q^*\|_\infty \leq \gamma^n \|Q_0 - Q^*\|_\infty$

### Step 3: DQN Approximation Error

With a neural network $Q_\theta$, the DQN loss:

$$\mathcal{L}(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}}\left[(r + \gamma \max_{a'} Q_{\theta^-}(s',a') - Q_\theta(s,a))^2\right]$$

**Error decomposition:**

$$\|Q_\theta - Q^*\| \leq \underbrace{\|Q_\theta - \hat{Q}\|}_{\text{optimization error}} + \underbrace{\|\hat{Q} - \hat{Q}^*\|}_{\text{statistical error}} + \underbrace{\|\hat{Q}^* - Q^*\|}_{\text{approximation error}}$$

- **Optimization error:** Reduced by sufficient gradient steps
- **Statistical error:** Reduced by larger replay buffer $|\mathcal{D}|$
- **Approximation error:** Depends on network capacity (width/depth)

### Step 4: Target Network Stability

Without a target network ($\theta^- = \theta$), DQN training is unstable because the target moves with each update.

**Proof of instability (semi-gradient divergence):**

Consider the update $\theta \leftarrow \theta + \alpha(r + \gamma \max_{a'} Q_\theta(s',a') - Q_\theta(s,a)) \nabla_\theta Q_\theta(s,a)$.

The gradient of the full loss includes $\nabla_\theta(r + \gamma \max_{a'} Q_\theta(s',a'))$, but this is ignored (semi-gradient). The missing term can cause the parameters to diverge when $\gamma$ is large.

The target network ($\theta^-$ updated periodically) stabilizes training by making the target approximately stationary between updates. $\blacksquare$

### Step 5: Color Action Space Efficiency

Our 12 actions span RGB gains, HSV adjustments, and warm/cool shifts. The **effective dimensionality** of the action space:

$$d_{\text{eff}} = \text{rank}(\mathbf{J}_a) \quad \text{where } J_{ij} = \frac{\partial I_i}{\partial a_j}$$

RGB gains contribute 3 independent dimensions, hue shift adds 1, saturation adds 1, warm/cool approximates 2 (R/B shift). Total: $d_{\text{eff}} \approx 5$-$6$ out of 12 actions.

The Q-network must learn which actions are redundant and focus on the effective subspace. $\blacksquare$

## 7 — Training Curves

In [ ]:
def moving_avg(data, w=20):
    return np.convolve(data, np.ones(w)/w, mode="valid")

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].plot(episode_rewards, alpha=0.3, color="steelblue")
axes[0].plot(moving_avg(episode_rewards), color="navy", lw=2)
axes[0].set(xlabel="Episode", ylabel="Reward", title="Episode Reward")
axes[0].grid(True, alpha=0.3)

axes[1].plot(episode_psnrs, alpha=0.3, color="coral")
axes[1].plot(moving_avg(episode_psnrs), color="darkred", lw=2)
axes[1].set(xlabel="Episode", ylabel="PSNR (dB)", title="Final PSNR")
axes[1].grid(True, alpha=0.3)

axes[2].plot(episode_des, alpha=0.3, color="orchid")
axes[2].plot(moving_avg(episode_des), color="purple", lw=2)
axes[2].set(xlabel="Episode", ylabel="ΔE", title="Colour Error (ΔE)")
axes[2].grid(True, alpha=0.3)

axes[3].plot(losses[:600], alpha=0.3, color="mediumseagreen")
if len(losses) > 20:
    axes[3].plot(moving_avg(losses[:600]), color="darkgreen", lw=2)
axes[3].set(xlabel="Step", ylabel="Loss", title="DQN Huber Loss")
axes[3].grid(True, alpha=0.3)

plt.suptitle("Colour Correction Agent — Training Dynamics", fontsize=14)
plt.tight_layout()
plt.show()

## Deep Derivation: Perceptual Color Spaces and Gamut Mapping

### Step 1: Chromaticity Diagram and Gamut

The CIE chromaticity coordinates normalize out luminance:

$$x = \frac{X}{X + Y + Z}, \quad y = \frac{Y}{X + Y + Z}, \quad z = 1 - x - y$$

The sRGB gamut in chromaticity space forms a triangle with vertices at:
- Red primary: $(x_R, y_R) = (0.64, 0.33)$
- Green primary: $(x_G, y_G) = (0.30, 0.60)$
- Blue primary: $(x_B, y_B) = (0.15, 0.06)$

**Proof gamut is convex:** Any color $(x,y)$ inside the gamut is a convex combination of primaries: $(x,y) = \alpha(x_R, y_R) + \beta(x_G, y_G) + (1-\alpha-\beta)(x_B, y_B)$ with $\alpha, \beta \geq 0$, $\alpha + \beta \leq 1$. Convex combination of convex set → convex. $\blacksquare$

### Step 2: Gamut Clipping During Color Correction

When the RL agent applies gain corrections, colors may leave the valid gamut. Clipping strategies:

**Per-channel clipping:** $I'_c = \max(0, \min(1, g_c \cdot I_c))$ — simple but causes hue shifts.

**Proof per-channel clipping changes hue:**

Original: $(R, G, B) = (0.8, 0.4, 0.2)$, hue $h = 30°$.
After $2\times$ gain: $(1.6, 0.8, 0.4)$. Clipped: $(1.0, 0.8, 0.4)$.
New hue: $h' = 60° \cdot \frac{0.8 - 0.4}{1.0 - 0.4} = 40° \neq 30°$. $\blacksquare$

**Gamut mapping via desaturation:** Project toward the neutral axis while preserving hue. Reduce chroma until the color is in-gamut.

### Step 3: Color Reproduction Index (CRI)

$$\text{CRI} = \frac{1}{8}\sum_{i=1}^{8} R_i, \quad R_i = 100 - 4.6 \Delta E_i$$

where $\Delta E_i$ is the color difference of test color sample $i$ under test vs. reference illuminant. CRI = 100 means perfect color rendering.

**Connection to RL reward:** The agent's reward combines PSNR and $\Delta E$. High CRI illuminant → low $\Delta E$ → high reward. The agent learns to produce corrections that maximize CRI indirectly. $\blacksquare$

## 8 — Before / After Evaluation

In [ ]:
def evaluate_colour_agent(env, policy_net, n=6):
    results = []
    policy_net.eval()
    for _ in range(n):
        obs, info = env.reset()
        initial = env.current_img.copy()
        init_psnr, init_de = info["psnr"], info["delta_e"]
        for t in range(MAX_STEPS):
            with torch.no_grad():
                action = policy_net(torch.FloatTensor(obs).unsqueeze(0).to(device)).argmax(1).item()
            obs, _, done, _, info = env.step(action)
            if done:
                break
        results.append({"cast": initial, "corrected": env.current_img.copy(), "target": env.target_img.copy(),
                        "psnr_before": init_psnr, "psnr_after": info["psnr"],
                        "de_before": init_de, "de_after": info["delta_e"]})
    policy_net.train()
    return results

results = evaluate_colour_agent(env, policy_net)

fig, axes = plt.subplots(3, 6, figsize=(20, 10))
for i, r in enumerate(results):
    axes[0, i].imshow(r["cast"])
    axes[0, i].set_title(f"Cast\nΔE={r['de_before']:.1f}", fontsize=9)
    axes[0, i].axis("off")
    axes[1, i].imshow(np.clip(r["corrected"], 0, 1))
    axes[1, i].set_title(f"Corrected\nΔE={r['de_after']:.1f}", fontsize=9)
    axes[1, i].axis("off")
    axes[2, i].imshow(r["target"])
    axes[2, i].set_title("Target", fontsize=9)
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Colour Cast", fontsize=11)
axes[1, 0].set_ylabel("Agent Output", fontsize=11)
axes[2, 0].set_ylabel("Ground Truth", fontsize=11)
plt.suptitle("Colour Correction Agent — Before vs After", fontsize=14)
plt.tight_layout()
plt.show()

print(f"{'Img':>4} | {'PSNR Before':>11} | {'PSNR After':>10} | {'ΔE Before':>9} | {'ΔE After':>8}")
print("-" * 56)
for i, r in enumerate(results):
    print(f"{i:4d} | {r['psnr_before']:11.2f} | {r['psnr_after']:10.2f} | {r['de_before']:9.2f} | {r['de_after']:8.2f}")

## 9 — LAB Space Analysis

Visualise the effect of correction in CIE-LAB space.

In [ ]:
def plot_lab_channels(img, title):
    lab = cv2.cvtColor((np.clip(img, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2LAB)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ch, name, cmap in zip(range(3), ["L*", "a*", "b*"], ["gray", "RdYlGn_r", "PuOr"]):
        axes[ch].imshow(lab[:, :, ch], cmap=cmap)
        axes[ch].set_title(f"{name}")
        axes[ch].axis("off")
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

r = results[0]
plot_lab_channels(r["cast"], "LAB Channels — Colour-Cast Image")
plot_lab_channels(r["corrected"], "LAB Channels — Agent-Corrected Image")
plot_lab_channels(r["target"], "LAB Channels — Ground Truth")

## 10 — Key Takeaways

| Insight | Detail |
|---------|--------|
| **Multi-space actions** | Mixing RGB gains, HSV hue/saturation, and warm/cool shifts gives the agent diverse correction tools |
| **ΔE metric** | Perceptually uniform colour error in LAB provides a more meaningful reward signal than raw RGB MSE |
| **Angular error** | Useful for evaluating illuminant estimation quality independent of brightness |
| **Exploration challenge** | 12 actions across colour spaces require longer training; curriculum learning could help |
| **Extensions** | Spatially-varying colour casts, real camera WB datasets, actor-critic for continuous gain control |

---

*Next notebook → Module 7.3: Denoising Agent with selective filter application.*